# 02 Scaled Dot-Product Attention (PyTorch Implementation & Plotting)

This notebook demonstrates **Scaled Dot-Product Attention** using PyTorch float tensors and visualizes how scaling by $\sqrt{d_k}$ prevents softmax logit saturation.

## Part 1: PyTorch Scaled Dot-Product Function
Define a reusable PyTorch function that computes $QK^T$, scales by $\sqrt{d_k}$, applies softmax, and computes weighted values $AV$.

In [ ]:
import torch
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

plt.style.use('dark_background')
torch.manual_seed(42)

def scaled_dot_product_attention(q, k, v, scale=True):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1))
    if scale:
        scores = scores / math.sqrt(d_k)
    attn_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, v)
    return output, attn_weights

q = torch.randn(1, 1, 10, 128)
k = torch.randn(1, 1, 10, 128)
v = torch.randn(1, 1, 10, 128)

out_scaled, weights_scaled = scaled_dot_product_attention(q, k, v, scale=True)
print('PyTorch Scaled Output Shape:', out_scaled.shape)
print('PyTorch Scaled Attention Weights Shape:', weights_scaled.shape)

## Part 2: Visualizing Softmax Logit Saturation (Unscaled vs Scaled)
Comparing unscaled attention logits ($QK^T$) with scaled logits ($QK^T / \sqrt{d_k}$) when $d_k=128$.

In [ ]:
_, unscaled_weights = scaled_dot_product_attention(q, k, v, scale=False)
_, scaled_weights = scaled_dot_product_attention(q, k, v, scale=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
im1 = ax1.imshow(unscaled_weights.squeeze().numpy(), cmap='magma', vmin=0, vmax=1)
ax1.set_title('Unscaled Attention (d_k=128)\nSoftmax Saturated (Vanishing Gradients)', color='#ff6b6b')
fig.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(scaled_weights.squeeze().numpy(), cmap='viridis', vmin=0, vmax=0.5)
ax2.set_title('Scaled Attention (d_k=128, factor 1/sqrt(128))\nActive Softmax (Healthy Gradients)', color='#51cf66')
fig.colorbar(im2, ax=ax2, shrink=0.8)
plt.tight_layout()
plt.show()